# Bug Prediction System — Final Model Training
**Dataset:** combined_output.csv
- 16,722 commits (Python + TypeScript)
- 873 bugs (5.22% bug rate)
- 3 time periods: 2018-2020, 2021-2023, 2024-2026
- No missing values ✅

**Models:** Random Forest vs XGBoost → best wins
---

## Step 1 — Install & Import

In [ ]:
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib shap -q
print('Done')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    precision_score, recall_score, f1_score, classification_report
)
from xgboost import XGBClassifier
import shap
print('Imports done')

## Step 2 — Load & Inspect Data

In [ ]:
df = pd.read_csv('combined_output.csv')

print('=== DATASET OVERVIEW ===')
print(f'Total commits   : {len(df):,}')
print(f'Total bugs      : {df["is_buggy"].sum()}')
print(f'Bug rate        : {df["is_buggy"].mean():.2%}')
print(f'Missing values  : {df.isnull().sum().sum()}')
print(f'Columns         : {len(df.columns)}')

print(f'\n=== BY LANGUAGE ===')
lang = df.groupby('language_group')['is_buggy'].agg(['count','sum','mean'])
lang.columns = ['commits','bugs','bug_rate']
print(lang.round(4))

print(f'\n=== BY TIME PERIOD ===')
time = df.groupby('time_period')['is_buggy'].agg(['count','sum','mean'])
time.columns = ['commits','bugs','bug_rate']
print(time.round(4))

## Step 3 — Feature Engineering

In [ ]:
# Encode language_group → number
le_lang   = LabelEncoder()
le_period = LabelEncoder()

df['lang_enc']   = le_lang.fit_transform(df['language_group'])
df['period_enc'] = le_period.fit_transform(df['time_period'])

print('Language encoding:')
for i, c in enumerate(le_lang.classes_):
    print(f'  {c} -> {i}')

print('\nTime period encoding:')
for i, c in enumerate(le_period.classes_):
    print(f'  {c} -> {i}')

# ── Feature columns ──────────────────────────────────────────
FEATURES = [
    # Strongest predictors
    'prior_bugs_author',    # developer bug history
    'avg_complexity',       # code complexity
    'test_ratio',           # test coverage in commit
    'test_files_changed',   # raw test file count
    'complexity_per_file',  # complexity density
    # Moderate predictors
    'files_changed',
    'num_methods',
    'churn_ratio',
    'lines_added',
    'lines_deleted',
    # Timing (weak but real)
    'commit_hour',
    'day_of_week',
    'is_weekend',
    'is_night_commit',
    # Research features
    'lang_enc',             # Python=0, TypeScript=1
    'period_enc',           # era encoding
]

X = df[FEATURES].fillna(0)
y = df['is_buggy']

print(f'\nFeature matrix : {X.shape}')
print(f'Bugs           : {y.sum()} ({y.mean():.2%})')

## Step 4 — EDA Charts

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Bug Prediction — Data Overview (Python + TypeScript | 16,722 commits)',
             fontsize=13, fontweight='bold')

# 1. Class balance
ax = axes[0, 0]
counts = y.value_counts().sort_index()
bars = ax.bar(['Clean', 'Buggy'], counts.values,
              color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
ax.set_title('Class Balance', fontweight='bold')
ax.set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+30, f'{val:,}',
            ha='center', fontweight='bold', fontsize=11)

# 2. Bug rate by language
ax = axes[0, 1]
lang_bug = df.groupby('language_group')['is_buggy'].mean()
bars2 = ax.bar(lang_bug.index, lang_bug.values,
               color=['#3498db', '#e67e22'], edgecolor='white', linewidth=1.5)
ax.set_title('Bug Rate by Language', fontweight='bold')
ax.set_ylabel('Bug Rate')
for bar, val in zip(bars2, lang_bug.values):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.001, f'{val:.2%}',
            ha='center', fontweight='bold')

# 3. Bug rate by time period
ax = axes[0, 2]
time_bug = df.groupby('time_period')['is_buggy'].mean().sort_index()
bars3 = ax.bar(time_bug.index, time_bug.values,
               color=['#3498db','#e67e22','#9b59b6'], edgecolor='white')
ax.set_title('Bug Rate by Time Period', fontweight='bold')
ax.set_ylabel('Bug Rate')
ax.tick_params(axis='x', rotation=15)
for bar, val in zip(bars3, time_bug.values):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.001, f'{val:.2%}',
            ha='center', fontweight='bold')

# 4. Correlation with is_buggy
ax = axes[1, 0]
corr = X.corrwith(y).sort_values(ascending=False)
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.set_title('Feature Correlation with is_buggy', fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation')

# 5. Prior bugs author distribution
ax = axes[1, 1]
prior = df.groupby('prior_bugs_author')['is_buggy'].mean().head(12)
ax.bar(prior.index.astype(str), prior.values,
       color='#9b59b6', edgecolor='white')
ax.set_title('Bug Rate by Prior Bugs (Author)', fontweight='bold')
ax.set_xlabel('Prior bugs this author caused')
ax.set_ylabel('Bug Rate')

# 6. Lines added distribution
ax = axes[1, 2]
clean_l = df[df['is_buggy']==0]['lines_added'].clip(0, 300)
buggy_l = df[df['is_buggy']==1]['lines_added'].clip(0, 300)
ax.hist(clean_l, bins=30, alpha=0.6, color='#2ecc71', label='Clean')
ax.hist(buggy_l, bins=30, alpha=0.7, color='#e74c3c', label='Buggy')
ax.set_title('Lines Added (clipped at 300)', fontweight='bold')
ax.set_xlabel('Lines Added')
ax.legend()

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_overview.png')

## Step 5 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y       # keeps 5.22% bug rate in both sets
)

print(f'Training set  : {len(X_train):,} commits')
print(f'  Bugs        : {y_train.sum()} ({y_train.mean():.2%})')
print(f'  Clean       : {(y_train==0).sum():,}')
print(f'\nTest set      : {len(X_test):,} commits')
print(f'  Bugs        : {y_test.sum()} ({y_test.mean():.2%})')
print(f'  Clean       : {(y_test==0).sum():,}')

## Step 6 — Train Random Forest

In [ ]:
rf_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(
                    n_estimators=200,
                    class_weight='balanced',  # handles 5.22% bug rate
                    max_depth=10,
                    min_samples_leaf=5,
                    random_state=42,
                    n_jobs=-1
                ))
])

rf_model.fit(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_cv = cross_val_score(rf_model, X_train, y_train,
                        cv=cv, scoring='roc_auc')

print('Random Forest trained')
print(f'  CV AUC scores : {rf_cv.round(3)}')
print(f'  CV AUC mean   : {rf_cv.mean():.3f} +/- {rf_cv.std():.3f}')

## Step 7 — Train XGBoost

In [ ]:
scale = (y_train==0).sum() / (y_train==1).sum()
print(f'scale_pos_weight = {scale:.1f}')

xgb_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     XGBClassifier(
                    n_estimators=200,
                    max_depth=5,
                    learning_rate=0.05,
                    scale_pos_weight=scale,
                    random_state=42,
                    eval_metric='auc',
                    verbosity=0
                ))
])

xgb_model.fit(X_train, y_train)

xgb_cv = cross_val_score(xgb_model, X_train, y_train,
                         cv=cv, scoring='roc_auc')

print('\nXGBoost trained')
print(f'  CV AUC scores : {xgb_cv.round(3)}')
print(f'  CV AUC mean   : {xgb_cv.mean():.3f} +/- {xgb_cv.std():.3f}')

## Step 8 — Pick Best Model & Save

In [ ]:
print('=== MODEL COMPARISON ===')
print(f'Random Forest CV AUC : {rf_cv.mean():.3f}')
print(f'XGBoost       CV AUC : {xgb_cv.mean():.3f}')

if rf_cv.mean() >= xgb_cv.mean():
    best_model = rf_model
    best_name  = 'Random Forest'
else:
    best_model = xgb_model
    best_name  = 'XGBoost'

print(f'\nWinner : {best_name}')

# Save everything needed for deployment
joblib.dump(best_model, 'bug_prediction_model.pkl')
joblib.dump(le_lang,    'encoder_language.pkl')
joblib.dump(le_period,  'encoder_period.pkl')
joblib.dump(FEATURES,   'feature_cols.pkl')

print('Saved:')
print('  bug_prediction_model.pkl')
print('  encoder_language.pkl')
print('  encoder_period.pkl')
print('  feature_cols.pkl')

## Step 9 — Evaluate on Test Set

In [ ]:
y_proba = best_model.predict_proba(X_test)[:, 1]
y_pred  = best_model.predict(X_test)

auc       = roc_auc_score(y_test, y_proba)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)

print('=' * 50)
print(f'  TEST RESULTS — {best_name}')
print('=' * 50)
print(f'  AUC-ROC   : {auc:.3f}')
print(f'  Precision : {precision:.3f}  (of commits flagged, how many real bugs)')
print(f'  Recall    : {recall:.3f}  (of real bugs, how many caught)')
print(f'  F1 Score  : {f1:.3f}')
print('=' * 50)
print()
print(classification_report(y_test, y_pred,
      target_names=['Clean','Buggy'], zero_division=0))

## Step 10 — Evaluation Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Model Evaluation — {best_name}', fontsize=14, fontweight='bold')

# ROC Curve
ax = axes[0]
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax.plot(fpr, tpr, color='#e74c3c', linewidth=2.5,
        label=f'AUC = {auc:.3f}')
ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random (0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
ax.set_title('ROC Curve', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()

# Confusion Matrix
ax = axes[1]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax,
            xticklabels=['Pred Clean','Pred Buggy'],
            yticklabels=['Actual Clean','Actual Buggy'],
            linewidths=0.5)
ax.set_title('Confusion Matrix', fontweight='bold')

# Feature Importance
ax = axes[2]
clf = best_model.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    imp = clf.feature_importances_
    idx = np.argsort(imp)
    ax.barh([FEATURES[i] for i in idx],
            imp[idx], color='#3498db', edgecolor='white')
    ax.set_title('Feature Importance', fontweight='bold')
    ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_evaluation.png')

## Step 11 — SHAP Explanation

In [ ]:
X_test_proc = best_model[:-1].transform(X_test)
clf         = best_model.named_steps['clf']

explainer   = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_proc)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test_proc,
                  feature_names=FEATURES,
                  show=False, plot_type='bar')
plt.title('SHAP — What drives bug predictions?', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_importance.png')

## Step 12 — Predict Any New Commit

In [ ]:
def predict_commit(commit_dict):
    """
    Predict bug risk for any new commit.
    Pass a dict with feature values.
    Returns probability and risk label.
    """
    # Encode language
    lang = commit_dict.get('language_group', 'Python')
    commit_dict['lang_enc'] = (
        le_lang.transform([lang])[0]
        if lang in le_lang.classes_ else 0
    )

    # Encode time period
    period = commit_dict.get('time_period', '2024-2026')
    commit_dict['period_enc'] = (
        le_period.transform([period])[0]
        if period in le_period.classes_ else 2
    )

    row  = pd.DataFrame([{col: commit_dict.get(col, 0) for col in FEATURES}])
    prob = best_model.predict_proba(row)[0][1]

    risk = ('HIGH RISK'   if prob >= 0.6 else
            'MEDIUM RISK' if prob >= 0.3 else
            'LOW RISK')

    icon = ('🔴' if prob >= 0.6 else
            '🟡' if prob >= 0.3 else '🟢')

    print(f'  Bug Probability : {prob:.1%}')
    print(f'  Risk Level      : {icon} {risk}')
    return prob, risk


# ── Example 1: Risky commit ──────────────────────────────────
print('=== Example 1: Risky Commit ===')
predict_commit({
    'lines_added'        : 800,
    'lines_deleted'      : 200,
    'files_changed'      : 25,
    'avg_complexity'     : 15.0,
    'num_methods'        : 30,
    'test_files_changed' : 0,
    'test_ratio'         : 0.0,
    'complexity_per_file': 0.6,
    'churn_ratio'        : 4.0,
    'prior_bugs_author'  : 8,
    'commit_hour'        : 23,
    'day_of_week'        : 6,
    'is_weekend'         : 1,
    'is_night_commit'    : 1,
    'language_group'     : 'Python',
    'time_period'        : '2024-2026',
})

print()

# ── Example 2: Safe commit ───────────────────────────────────
print('=== Example 2: Safe Commit ===')
predict_commit({
    'lines_added'        : 12,
    'lines_deleted'      : 3,
    'files_changed'      : 2,
    'avg_complexity'     : 1.5,
    'num_methods'        : 2,
    'test_files_changed' : 2,
    'test_ratio'         : 1.0,
    'complexity_per_file': 0.5,
    'churn_ratio'        : 4.0,
    'prior_bugs_author'  : 0,
    'commit_hour'        : 10,
    'day_of_week'        : 1,
    'is_weekend'         : 0,
    'is_night_commit'    : 0,
    'language_group'     : 'TypeScript',
    'time_period'        : '2024-2026',
})

## Step 13 — Final Summary

In [ ]:
clf = best_model.named_steps['clf']

print('=' * 55)
print('  BUG PREDICTION MODEL — FINAL SUMMARY')
print('=' * 55)
print(f'  Dataset         : 16,722 real GitHub commits')
print(f'  Languages       : Python + TypeScript')
print(f'  Time span       : 2018 to 2026')
print(f'  Bug count       : 873 ({df["is_buggy"].mean():.2%})')
print(f'  Features used   : {len(FEATURES)}')
print(f'  Best model      : {best_name}')
print(f'  Test AUC        : {auc:.3f}')
print(f'  Precision       : {precision:.3f}')
print(f'  Recall          : {recall:.3f}')
print(f'  F1 Score        : {f1:.3f}')
print('=' * 55)

if hasattr(clf, 'feature_importances_'):
    top3 = np.argsort(clf.feature_importances_)[-3:][::-1]
    print(f'\n  Top 3 Predictors:')
    for rank, idx in enumerate(top3):
        print(f'    {rank+1}. {FEATURES[idx]}'
              f'  (importance={clf.feature_importances_[idx]:.3f})')

print(f'\n  Saved files:')
for f in ['bug_prediction_model.pkl','encoder_language.pkl',
          'encoder_period.pkl','feature_cols.pkl',
          'eda_overview.png','model_evaluation.png','shap_importance.png']:
    print(f'    {f}')
print('=' * 55)